# Desafio

## Análise de Desmatamento no Cerrado

Você trabalha no INPE analisando dados do PRODES Cerrado. Recebeu um dataset com alertas de desmatamento de 2023 e 2024.

**Tarefa:**

1. Crie um DataFrame com 300 registros de alertas com as colunas:
   - `data` (datas entre 2023-01-01 e 2024-12-31)
   - `estado` (TO, BA, MA, PI, GO, MT — estados do Cerrado)
   - `municipio` (pelo menos 8 nomes diferentes de municípios)
   - `area_ha` (área desmatada, valores entre 1 e 500 hectares)
   - `uso_anterior` (Vegetação Nativa, Pastagem, Agricultura)
   
   Use `np.random.seed(42)` para reprodutibilidade.

2. Responda às seguintes perguntas:
   - Qual estado teve a maior área total desmatada?
   - Qual município teve mais alertas?
   - Qual o uso anterior mais associado a desmatamento?
   - Em qual trimestre de 2024 houve mais desmatamento?

3. Crie uma coluna `taxa_ha_por_dia` = `area_ha` / dias desde início da série

4. Exporte um resumo por estado (total de área, média, número de alertas) para CSV

**Dica:** Use `groupby()`, `agg()`, `resample()`, `nlargest()`, `value_counts()`.

### Criando o dataframe

In [106]:
import pandas as pd
import numpy as np

In [107]:
np.random.seed(42)
n = 300

municipios = {"Goiânia" : "GO", "Cuiabá ": "MT", "Palmas": "TO",
    "Anápolis": "GO", "Rondonópolis": "MT", "Barreiras": "BA",
    "Uruçuí": "PI", "Balsas": "MA"}

#dataframe com 300 registros de alerta
df = pd.DataFrame({
    "data": pd.date_range(start="2023-01-01", end="2024-12-31", periods=n).date,
    "municipio": np.random.choice(
        ["Goiânia", "Cuiabá ", "Palmas", "Anápolis",
        "Rondonópolis", "Barreiras", "Uruçuí", "Balsas"],
        n,),
    "area_ha": np.random.triangular(left=1, mode=1, right=500, size=300),
    "uso_anterior": np.random.choice(
        ["Vegetação Nativa", "Pastagem", "Agricultura"],
        n)
})

df["estado"] = df.municipio.map(municipios)

df

,data,municipio,area_ha,uso_anterior,estado
0,2023-01-01,Uruçuí,348.864676,Vegetação Nativa,PI
1,2023-01-03,Anápolis,64.856518,Pastagem,GO
2,2023-01-05,Rondonópolis,38.565246,Agricultura,MT
3,2023-01-08,Uruçuí,143.451584,Agricultura,PI
4,2023-01-10,Palmas,440.224995,Pastagem,TO
...,...,...,...,...,...
295,2024-12-21,Anápolis,385.391287,Vegetação Nativa,GO
296,2024-12-23,Barreiras,440.959767,Agricultura,BA
297,2024-12-26,Uruçuí,252.191448,Vegetação Nativa,PI
298,2024-12-28,Rondonópolis,105.903582,Pastagem,MT


### Qual estado teve a maior área total desmatada

In [108]:
estados_area = df.groupby("estado").agg(area_total=("area_ha", "sum"))
area_maior = estados_area["area_total"].idxmax()
area_num = estados_area.loc["GO"]["area_total"]

print(f"O estado que teve a maior área desmatada foi {area_maior} com {area_num:.2f} hectares devastados.")

O estado que teve a maior área desmatada foi GO com 14847.04 hectares devastados.


### Qual município teve mais alertas?

In [109]:
municipios_alertas = df.groupby("municipio").agg(
    numero_alertas=("area_ha", "count"),
    area_total=("area_ha", "sum"),
    area_media=("area_ha", "mean"))

city_maior = municipios_alertas["numero_alertas"].idxmax()

print(f"A cidade mais desmatada foi {city_maior} com um total de {municipios_alertas.loc[city_maior]["numero_alertas"]} alertas.")

A cidade mais desmatada foi Anápolis com um total de 46.0 alertas.


### Qual o uso anterior mais associado a desmatamento?

In [110]:
usos = df.groupby("uso_anterior").agg(
    numero_alertas=("area_ha", "count"),
    area_total=("area_ha", "sum"),
    area_media=("area_ha", "mean")
)

maior_uso = usos["numero_alertas"].idxmax()

print(f"O uso anterior mais associado foi {maior_uso}, com uma devastação de {usos.loc[maior_uso]["area_total"]:.2f} hectares.")

O uso anterior mais associado foi Vegetação Nativa, com uma devastação de 19080.47 hectares.


### Em qual trimestre de 2024 houve mais desmatamento?

In [181]:
#selecionando apenas o ano de 2024
df_2024 = df.loc[df["data"].dt.year == 2024]

#mudando o index para o tipo datetime
df_2024["data"] = pd.to_datetime(df["data"])
df_time = df_2024.set_index("data")


trimestral = df_time.resample("QE").agg(
    alertas=("area_ha", "count"),
    area_total=("area_ha", "sum"))

maior_trimestre = trimestral["area_total"].idxmax()

print(f"O trimestre {maior_trimestre.date()} foi o que apresentou mais desmatamento, cerca de {trimestral.loc[maior_trimestre]["area_total"]:.2f} de hectares.")

O trimestre 2024-09-30 foi o que apresentou mais desmatamento, cerca de 7258.10 de hectares.


### Crie uma coluna `taxa_ha_por_dia` = `area_ha` / dias desde início da série

In [193]:
df["taxa_ha_por_dia"] = df["area_ha"] / 731
df

,data,municipio,area_ha,uso_anterior,estado,taxa_ha_por_dia
0,2023-01-01,Uruçuí,348.864676,Vegetação Nativa,PI,0.477243
1,2023-01-03,Anápolis,64.856518,Pastagem,GO,0.088723
2,2023-01-05,Rondonópolis,38.565246,Agricultura,MT,0.052757
3,2023-01-08,Uruçuí,143.451584,Agricultura,PI,0.196240
4,2023-01-10,Palmas,440.224995,Pastagem,TO,0.602223
...,...,...,...,...,...,...
295,2024-12-21,Anápolis,385.391287,Vegetação Nativa,GO,0.527211
296,2024-12-23,Barreiras,440.959767,Agricultura,BA,0.603228
297,2024-12-26,Uruçuí,252.191448,Vegetação Nativa,PI,0.344995
298,2024-12-28,Rondonópolis,105.903582,Pastagem,MT,0.144875


### Exporte um resumo por estado (total de área, média, número de alertas) para CSV

In [ ]:
estados_resumo = df.groupby("estado").agg(
    alertas=("area_ha", "count"),
    area_total=("area_ha", "sum"),
    media_area=("area_ha", "mean")
)

estados_resumo = estados_resumo.reset_index()

estados_resumo.to_csv("estados_resumo.csv", index=False)